## 필독!!!

<h3> 여기 있는 코드는 절대 실행하지 마십시오. </h3>

눈으로만 보고 이해하시거나

복붙하셔서 실제 linux나 파이썬 환경에서 실행해 주시기 바랍니다.

이 파일은 jupyter 파일입니다.

여기 있는 코드는 모두 jupyter가 아닌 실제 파이썬 및 ROS2 환경에서 사용할 수 있는 코드로 작성하였습니다.

ROS2 코드를 jupyter에서 실행하는 방법이 없는 것은 아니나 별도의 방법이 따로 존재하기 때문에(`jupyter_ws` 참고)

여기 있는 코드를 실행하게 될 경우 일부 오류나 무한루프 등에 빠질 수 있는 위험이 있습니다.

### Executor 패키지 실습

#### 1 - 1. SingleThreadedExecutor

(py_executor_example/py_executor_example/single_executor_node.py 참고.)

In [ ]:
import time
import rclpy
from rclpy.node import Node
from rclpy.executors import SingleThreadedExecutor

class SingleExecutorNode(Node):
    def __init__(self):
        super().__init__('single_executor_node')
        self.timer1 = self.create_timer(1.0, self.timer1_callback)          # 콜백함수 1
        self.timer2 = self.create_timer(1.0, self.timer2_callback)          # 콜백함수 2

    def timer1_callback(self):                                              # 콜백함수 1 - 3초 작업
        self.get_logger().info('timer1 시작 - 3초 작업')
        time.sleep(3)
        self.get_logger().info('timer1 종료')

    def timer2_callback(self):                                              # 콜백함수 2 
        self.get_logger().info('timer2 실행')               

def main(args=None):
    rclpy.init(args=args)
    node = SingleExecutorNode()                                             # 노드 생성
    executor = SingleThreadedExecutor()                                     # 싱글 스레드 실행기 생성
    executor.add_node(node)                                                 # 노드를 실행기에 추가

    try:
        executor.spin()
    except KeyboardInterrupt:
        pass
    finally:
        executor.shutdown()
        node.destroy_node()
        rclpy.shutdown()

if __name__ == '__main__':
    main()

구조가 상당히 유사한 부분이 많으므로 설명은 주석으로 처리했다.

main함수 부분을 굳이 try-except 구문으로 처리한 이유는

try-except 없이 코드를 짰다면 ctrl + C 로 프로세스를 종료 시켰을 때

finally 부분의 `executor.shutdown()`, `node.destroy_node()`, `rclpy.shutdown()`가 실행되지 않을 수 있기 때문이다.

#### 1 - 2. setup.py 수정 및 실행

In [ ]:
entry_points={
    'console_scripts': [
        'single_executor_node = py_executor_example.single_executor_node:main'
    ],
}

의 내용을 추가해준다.

#### 1 - 3. 빌드 및 실행

```bash
colcon build --packages-select py_executor_example
```

터미널을 열고 빌드를 해 준 다음

```bash
source install/setup.bash
ros2 run py_executor_example single_executor_node
```

실행 후 결과를 확인한다.

timer2가 timer1 때문에 1초보다 늦게 실행됨을 확인할 수 있다.

#### 2. MultiThreadedExecutor

MutuallyExclusive 버전

In [ ]:
import time
import rclpy
from rclpy.node import Node
from rclpy.executors import MultiThreadedExecutor
from rclpy.callback_groups import MutuallyExclusiveCallbackGroup

class MultiExecutorNode(Node):
    def __init__(self):
        super().__init__('multi_executor_node')
        self.callback_group1 = MutuallyExclusiveCallbackGroup()                 # 콜백 그룹 1 정의
        self.callback_group2 = MutuallyExclusiveCallbackGroup()                 # 콜백 그룹 2 정의

        self.timer1 = self.create_timer(
            1.0,
            self.timer1_callback,
            callback_group=self.callback_group1                                 # 콜백 그룹 1에 할당
        )

        self.timer2 = self.create_timer(
            1.0,
            self.timer2_callback,
            callback_group=self.callback_group2                                 # 콜백 그룹 2에 할당
        )

    def timer1_callback(self):
        self.get_logger().info('timer1 시작 - 3초 작업')
        time.sleep(3)
        self.get_logger().info('timer1 종료')

    def timer2_callback(self):
        self.get_logger().info('timer2 실행')


def main(args=None):
    rclpy.init(args=args)
    node = MultiExecutorNode()
    executor = MultiThreadedExecutor(num_threads=2)
    executor.add_node(node)
    try:
        executor.spin()
    except KeyboardInterrupt:
        pass
    finally:
        executor.shutdown()
        node.destroy_node()
        rclpy.shutdown()

if __name__ == '__main__':
    main()

#### 2 - 2. setup.py 수정 및 실행

In [ ]:
entry_points={
    'console_scripts': [
        'multi_exclusive_node = py_executor_example.multi_exclusive_node:main'
    ],
}

의 내용을 추가해준다.

#### 2 - 3. 빌드 및 실행

```bash
colcon build --packages-select py_executor_example
```

터미널을 열고 빌드를 해 준 다음

```bash
source install/setup.bash
ros2 run py_executor_example multi_exclusive_node
```

실행 후 결과를 확인한다.

timer2가 timer1과 상관없이 독립적으로 결과를 출력하는 것을 볼 수 있다.

timer2와 timer1의 소속 그룹을 하나로 통일시켜 빌드하고 다시 실행하여 결과를 확인하는 것도 좋은 방법이다.

Reentrant 버전

(py_executor_example/py_executor_example/multi_executor_node.py 참고.)

In [ ]:
import time
import rclpy
from rclpy.node import Node
from rclpy.executors import MultiThreadedExecutor
from rclpy.callback_groups import ReentrantCallbackGroup

class MultiExecutorNode(Node):
    def __init__(self):
        super().__init__('multi_executor_node')
        self.callback_group = ReentrantCallbackGroup()
        self.timer1 = self.create_timer(
            1.0,
            self.timer1_callback,
            callback_group=self.callback_group
        )
        self.timer2 = self.create_timer(
            1.0,
            self.timer2_callback,
            callback_group=self.callback_group
        )

    def timer1_callback(self):
        self.get_logger().info('timer1 시작 - 3초 작업')
        time.sleep(3)
        self.get_logger().info('timer1 종료')

    def timer2_callback(self):
        self.get_logger().info('timer2 실행')

def main(args=None):
    rclpy.init(args=args)
    node = MultiExecutorNode()
    executor = MultiThreadedExecutor(num_threads=2)
    executor.add_node(node)
    try:
        executor.spin()
    except KeyboardInterrupt:
        pass
    finally:
        executor.shutdown()
        node.destroy_node()
        rclpy.shutdown()

if __name__ == '__main__':
    main()

#### 2 - 2. setup.py 수정 및 실행

In [ ]:
entry_points={
    'console_scripts': [
        'multi_executor_node = py_executor_example.multi_executor_node:main'
    ],
}

의 내용을 추가해준다.

#### 2 - 3. 빌드 및 실행

```bash
colcon build --packages-select py_executor_example
```

터미널을 열고 빌드를 해 준 다음

```bash
source install/setup.bash
ros2 run py_executor_example multi_executor_node
```

실행 후 결과를 확인한다.

결과가 많이 꼬여 나오는 것을 확인할 수 있다.

이는 reetrant가 timer1, timer2만 포함하는 것이 아닌, timer1 자기 자신의 다음 호출도 포함하기 때문에 벌어지는 일이다.

즉 timer1이 작동하고 3초 대기하는 동안, 1초가 흐른 후 timer1이 다시 재가동되어, 

그 안의 timer2가 작동하면서 동시에 재가동된 timer1 3초 대기 때문에

그 timer1이 다시 1초 뒤 호출되면서 다시 timer2가 작동하고 ...

이런 과정이 꼬인 것이다.

multi_executor_node.py의 내용을 아래 코드로 바꾸고 다시 빌드해서 ros2 명령어를 실행시켜 보자. 

In [ ]:
import time
import rclpy
from rclpy.node import Node
from rclpy.executors import MultiThreadedExecutor
from rclpy.callback_groups import ReentrantCallbackGroup

class MultiExecutorNode(Node):
    def __init__(self):
        super().__init__('multi_executor_node')
        self.callback_group = ReentrantCallbackGroup()
        self.timer1_running = False
        self.timer1 = self.create_timer(
            1.0,
            self.timer1_callback,
            callback_group=self.callback_group
        )

        self.timer2 = self.create_timer(
            1.0,
            self.timer2_callback,
            callback_group=self.callback_group
        )

    def timer1_callback(self):
        if self.timer1_running:                                                     # timer1이 이미 실행 중이면 이번 호출은 건너뜀
            self.get_logger().info('timer1 이미 실행 중이라 이번 호출은 건너뜀')
            return
        self.timer1_running = True                                                  # timer1의 실행 상태로 사용할 변수
        try:
            self.get_logger().info('timer1 시작 - 3초 작업')
            time.sleep(3)
            self.get_logger().info('timer1 종료')
        finally:
            self.timer1_running = False                                             # timer1 작업이 끝나면 실행 상태 초기화         

    def timer2_callback(self):
        self.get_logger().info('timer2 실행')

def main(args=None):
    rclpy.init(args=args)
    node = MultiExecutorNode()
    executor = MultiThreadedExecutor(num_threads=2)
    executor.add_node(node)

    try:
        executor.spin()
    except KeyboardInterrupt:
        pass
    finally:
        executor.shutdown()
        node.destroy_node()
        rclpy.shutdown()

if __name__ == '__main__':
    main()

문제점은 timer1이 실행중인데도 불구하고 다시 실행되는 점이었으므로

timer1에 실행중이면 더 실행하지 않도록 코드를 수정한 내용이다.

수정한 후 빌드 및 실행을 하면

timer1과 timer2가 독립적으로 잘 실행되는 것을 확인할 수 있다.